In [5]:
import os
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from io import BytesIO
import base64

# -------------------- CONFIG --------------------
feedback_file = r"D:\shree\Miniproject\voicetomusic\Voice2Music\output\comparitative_analysis_report\feedback.csv"
report_file = r"D:\shree\Miniproject\voicetomusic\Voice2Music\output\comparitative_analysis_report\model_comparative_report.html"

# Paths to model outputs (MIDI or symbolic representation)
model_outputs = {
    'PerformanceRNN': r"D:\shree\Miniproject\voicetomusic\Voice2Music\output\performance_rnn_rich_win\voice_music_rich.wav",
    'MusicVAE': r"D:\shree\Miniproject\voicetomusic\Voice2Music\output\music_vae\final_song_test.wav",
    # 'CustomModel': r'path\to\custom_model_outputs\'
}

models = list(model_outputs.keys()) + ['CustomModel']
metrics = ['melody_rating', 'accompaniment_rating', 'overall_rating']

sns.set_style('whitegrid')

# -------------------- LOAD OR CREATE CSV --------------------
def load_feedback():
    if not os.path.exists(feedback_file):
        df = pd.DataFrame(columns=['timestamp', 'model'] + metrics)
        df.to_csv(feedback_file, index=False)
        print('Created new feedback CSV.')
    return pd.read_csv(feedback_file)

# -------------------- GET USER FEEDBACK --------------------
def get_user_feedback():
    timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    entries = []

    print('\nPlease enter feedback for each model (1-5):\n')
    for m in models:
        if m == 'CustomModel':
            print('[NOTE] Custom model input is commented out until available.')
            # Uncomment when custom model is ready
            # melody = int(input(f'Melody rating for {m}: '))
            # accompaniment = int(input(f'Accompaniment rating for {m}: '))
            # overall = int(input(f'Overall rating for {m}: '))
            melody = accompaniment = overall = None
            entries.append([timestamp, m, melody, accompaniment, overall])
            continue

        while True:
            try:
                melody = int(input(f'Melody rating for {m} (1-5): '))
                accompaniment = int(input(f'Accompaniment rating for {m} (1-5): '))
                overall = int(input(f'Overall rating for {m} (1-5): '))
                if all(1 <= x <=5 for x in [melody, accompaniment, overall]):
                    break
                else:
                    print('Ratings must be between 1 and 5.')
            except ValueError:
                print('Enter valid integers between 1 and 5.')

        entries.append([timestamp, m, melody, accompaniment, overall])

    return entries
# -------------------- UPDATE CSV --------------------
def update_csv(entries):
    df = pd.read_csv(feedback_file)
    new_df = pd.DataFrame(entries, columns=['timestamp', 'model'] + metrics)
    df = pd.concat([df, new_df], ignore_index=True)
    df.to_csv(feedback_file, index=False)
    print('\nFeedback updated successfully.')

def main():
    print('=== Model Feedback ===')
    df = load_feedback()
    new_entries = get_user_feedback()
    update_csv(new_entries)

if __name__ == '__main__':
    main()

=== Model Feedback ===

Please enter feedback for each model (1-5):



Melody rating for PerformanceRNN (1-5):  3
Accompaniment rating for PerformanceRNN (1-5):  3.5


Enter valid integers between 1 and 5.


Melody rating for PerformanceRNN (1-5):  3
Accompaniment rating for PerformanceRNN (1-5):  3
Overall rating for PerformanceRNN (1-5):  3
Melody rating for MusicVAE (1-5):  3
Accompaniment rating for MusicVAE (1-5):  4
Overall rating for MusicVAE (1-5):  4


[NOTE] Custom model input is commented out until available.

Feedback updated successfully.


In [12]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
import base64
import librosa
import matplotlib.gridspec as gridspec

# -------------------- CONFIG --------------------

feedback_file = r"D:\shree\Miniproject\voicetomusic\Voice2Music\output\comparitative_analysis_report\feedback.csv"
report_file = r"D:\shree\Miniproject\voicetomusic\Voice2Music\output\comparitative_analysis_report\model_comparative_report.html"

model_audio_paths = {
    "PerformanceRNN": r"D:\shree\Miniproject\voicetomusic\Voice2Music\output\performance_rnn_rich_win\voice_music_rich.wav",
    "MusicVAE": r"D:\shree\Miniproject\voicetomusic\Voice2Music\output\music_vae\final_song_test.wav",
}

metrics = ["melody_rating", "accompaniment_rating", "overall_rating"]

# -------------------- LOAD FEEDBACK --------------------

def load_feedback():
    if not os.path.exists(feedback_file):
        df = pd.DataFrame(columns=["timestamp", "model"] + metrics)
        df.to_csv(feedback_file, index=False)
    return pd.read_csv(feedback_file)

# -------------------- AUDIO METRICS --------------------

def extract_fast_metrics(path):
    try:
        y, sr = librosa.load(path, sr=22050)
        duration = librosa.get_duration(y=y, sr=sr)
        rms_energy = float(np.mean(librosa.feature.rms(y=y)))
        spectral_contrast = float(np.mean(librosa.feature.spectral_contrast(y=y, sr=sr)))
        harmonic_y = librosa.effects.harmonic(y)
        harmony_fit = float(np.mean(np.abs(harmonic_y)))
        onset_env = librosa.onset.onset_strength(y=y, sr=sr)
        tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0])
        onset_frames = librosa.onset.onset_detect(y=harmonic_y, sr=sr)
        note_density = len(onset_frames) / duration if duration > 0 else 0
        spectral_centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
        return {
            "duration": duration,
            "energy": rms_energy,
            "spectral_contrast": spectral_contrast,
            "harmony_fit": harmony_fit,
            "tempo": tempo,
            "note_density": note_density,
            "spectral_centroid": spectral_centroid
        }
    except Exception as e:
        print(f"Error processing {path}: {e}")
        return {"duration": 0, "energy": 0, "spectral_contrast": 0, "harmony_fit": 0,
                "tempo": 0, "note_density": 0, "spectral_centroid": 0}

def analyze_audio_models():
    results = {}
    for model, path in model_audio_paths.items():
        results[model] = extract_fast_metrics(path)
    return results

# -------------------- PLOT TO BASE64 --------------------

def plot_to_base64(fig):
    buf = BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=150)
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")

# -------------------- GENERATE HTML REPORT --------------------

def generate_html_report():
    df = load_feedback()
    audio_metrics = analyze_audio_models()
    feedback_avg = df.groupby("model")[metrics].mean()
    audio_df = pd.DataFrame(audio_metrics).T

    # --------- FEEDBACK BAR GRAPH ---------
    fig, ax = plt.subplots(figsize=(7, 4))
    feedback_avg.plot(kind="bar", ax=ax)
    ax.set_title("User Feedback Comparison", fontsize=13)
    ax.set_ylabel("Average Score")
    ax.set_ylim(0, 5)
    feedback_b64 = plot_to_base64(fig)

    # --------- AUDIO METRICS SIDE-BY-SIDE BAR GRAPHS WITH GAP ---------
    fig2 = plt.figure(figsize=(12, 4))
    gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], wspace=0.5)  # gap between plots

    # Left subplot: Large-scale metrics
    ax2 = fig2.add_subplot(gs[0])
    left_metrics = ['duration', 'spectral_contrast', 'tempo']
    bar_width = 0.25
    indices = np.arange(len(audio_df))
    colors_left = ['#1f77b4', '#ff7f0e', '#2ca02c']  # retain previous colors
    for i, metric in enumerate(left_metrics):
        ax2.bar(indices + i*bar_width, audio_df[metric], width=bar_width, label=metric, color=colors_left[i])
    ax2.set_title("Large-Scale Audio Metrics")
    ax2.set_xticks(indices + bar_width)
    ax2.set_xticklabels(audio_df.index)
    ax2.set_ylim(0, audio_df[left_metrics].max().max()*1.2)
    ax2.legend()

    # Right subplot: Small-scale metrics
    ax3 = fig2.add_subplot(gs[1])
    right_metrics = ['energy', 'harmony_fit', 'note_density', 'spectral_centroid']
    colors_right = ['#FF5733', '#33C1FF', '#FFD700', '#00FFAA']  # retain previous colors
    bar_width2 = 0.18
    indices2 = np.arange(len(audio_df))
    for i, metric in enumerate(right_metrics):
        ax3.bar(indices2 + i*bar_width2, audio_df[metric]/audio_df[metric].max(),
                width=bar_width2, color=colors_right[i], label=metric)
    ax3.set_title("Small-Scale Audio Metrics (Normalized)")
    ax3.set_xticks(indices2 + bar_width2*1.5)
    ax3.set_xticklabels(audio_df.index)
    ax3.set_ylim(0, 1.2)
    ax3.legend()

    buf2 = BytesIO()
    fig2.savefig(buf2, format="png", bbox_inches="tight", dpi=150)
    plt.close(fig2)
    audio_b64 = base64.b64encode(buf2.getvalue()).decode("utf-8")

    # --------- HTML CONTENT ---------
    html = f"""
    <html>
    <head>
        <title>Model Comparative Analysis Report</title>
        <style>
            body {{
                font-family: 'Segoe UI', sans-serif;
                background: linear-gradient(135deg, #001f54 0%, #9d00ff 100%);
                padding: 20px;
            }}
            .card {{
                background: rgba(255, 255, 255, 0.85);
                border-radius: 18px;
                padding: 25px;
                margin-bottom: 35px;
                backdrop-filter: blur(10px);
                box-shadow: 0 4px 25px rgba(0,0,0,0.2);
            }}
            h1 {{
                text-align: center;
                color: #ffffff;
                text-shadow: 0 0 10px rgba(0,0,0,0.4);
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin-top: 20px;
            }}
            th {{
                background: #4b0082;
                color: white;
                padding: 10px;
            }}
            td {{
                padding: 10px;
                background: #ffffff;
                text-align: center;
                border-bottom: 1px solid #ccc;
            }}
        </style>
    </head>

    <body>

        <h1>Model Comparative Analysis Report</h1>

        <div class="card">
            <h2>User Feedback Results</h2>
            <div style="margin-bottom:30px;">
                {feedback_avg.to_html(classes='table table-striped')}
            </div>
            <div style="text-align:center; margin-top:30px;">
                <img src="data:image/png;base64,{feedback_b64}" style="max-width:90%; border-radius:12px;">
            </div>
        </div>

        <div class="card">
            <h2>Audio-Based Evaluation Metrics</h2>
            <div style="margin-bottom:30px;">
                {audio_df.to_html(classes='table table-striped')}
            </div>
            <div style="text-align:center; margin-top:20px;">
                <img src="data:image/png;base64,{audio_b64}" style="max-width:90%; border-radius:12px;">
            </div>
        </div>

        <div class="card">
            <h2>Notes</h2>
            <p>• Metrics include Duration, RMS Energy, Spectral Contrast, Spectral Centroid, Harmony Fit, Tempo, and Note Density.</p>
            <p>• Feedback remains the primary evaluation of perceptual quality.</p>
            <p>• Audio metrics indicate alignment, harmonic consistency, rhythmic stability, and accompaniment richness.</p>
        </div>

    </body>
    </html>
    """

    os.makedirs(os.path.dirname(report_file), exist_ok=True)
    with open(report_file, "w", encoding="utf-8") as f:
        f.write(html)

    print("\n✨ Report successfully generated!")
    print("Saved to:", report_file)

# -------------------- RUN --------------------
generate_html_report()


C:\Users\All\AppData\Local\Temp\ipykernel_4068\2396332834.py:41: FutureWarning: librosa.beat.tempo
	This function was moved to 'librosa.feature.rhythm.tempo' in librosa version 0.10.0.
	This alias will be removed in librosa version 1.0.
  tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0])



✨ Report successfully generated!
Saved to: D:\shree\Miniproject\voicetomusic\Voice2Music\output\comparitative_analysis_report\model_comparative_report.html
